# Quickstart: загрузка обученной модели и прогноз

Минимальный пример, как взять один из 12 бандлов и получить прогноз для одного набора параметров процесса. Без torch / jax / dde — только numpy.

Зависимости для этого notebook'а: `numpy`, `matplotlib`, `pandas`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from models.loader import load_bundle, forward_predict
from physics.invariants import von_mises_stress
from visualization.style import MODEL_COLORS, COMPONENT_TEX_STRESS, apply_thesis_style

apply_thesis_style()

## Загрузка моделей

В `results/bundles/stress/` лежат шесть `.pkl` бандлов: MLP-baseline, MLP-grid, PINN-DeepXDE, PINN-JAX, PINN-PyTorch, VPINN. Все имеют одинаковый holdout split (305 наборов, seed=42).

In [ ]:
BUNDLES = {
    'MLP-grid':   '../results/bundles/stress/mlp_model_grid.pkl',
    'MLP-optuna': '../results/bundles/stress/mlp_model.pkl',
    'PINN-torch': '../results/bundles/stress/pinn_model_torch.pkl',
    'PINN-dde':   '../results/bundles/stress/pinn_model_dde.pkl',
    'PINN-jax':   '../results/bundles/stress/pinn_model_jax.pkl',
    'VPINN':      '../results/bundles/stress/vpinn_model.pkl',
}

bundles = {name: load_bundle(p) for name, p in BUNDLES.items()}
for name, b in bundles.items():
    print(f"{name:12s}  framework={b.get('framework', '?'):24s}  activation={b.get('activation', '?')}")

## Прогноз для одного набора параметров

Параметры процесса:  Q (редукция), k (упрочнение), α (полуугол матрицы, °), μ (трение), v (скорость, м/мин). Модель ждёт вектор `(Q, k, α, μ, v, r)` где `r ∈ [0, 1]` — нормированная радиальная координата.

In [ ]:
params = dict(Q=0.10, k=0.5, alpha=12.0, mu=0.05, v=20.0)

r_grid = np.linspace(0.0, 1.0, 20, dtype=np.float32)
X = np.zeros((20, 6), dtype=np.float32)
X[:, 0] = params['Q']
X[:, 1] = params['k']
X[:, 2] = params['alpha']
X[:, 3] = params['mu']
X[:, 4] = params['v']
X[:, 5] = r_grid

preds = {name: forward_predict(b, X) for name, b in bundles.items()}
for name, y in preds.items():
    vm = von_mises_stress(y)
    print(f"{name:12s}  σ_vm в МПа: min={vm.min():6.1f}  mean={vm.mean():6.1f}  max={vm.max():6.1f}")

## График — четыре компоненты тензора напряжений

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
axes = axes.ravel()

for ci in range(4):
    ax = axes[ci]
    for name, y in preds.items():
        ax.plot(r_grid, y[:, ci], label=name, color=MODEL_COLORS.get(name), lw=1.4)
    ax.set_title(COMPONENT_TEX_STRESS[ci])
    ax.set_xlabel(r'$r,\,-$')
    ax.set_ylabel('МПа')
    if ci == 0:
        ax.legend(fontsize=9)

fig.suptitle(f"Q={params['Q']*100:.0f}%, k={params['k']:.2f}, "
             f"α={params['alpha']:.0f}°, μ={params['mu']:.3f}, v={params['v']:.0f} м/мин",
             y=1.02)
plt.tight_layout()
plt.show()

## Что дальше

- `notebooks/01_data_exploration.ipynb` — обзор датасета FEM (требует X_stress.pkl / y_stress.pkl).
- `notebooks/04_physics_audit.ipynb` — проверка ГУ и уравнений равновесия на прогнозах.
- `python training/train_stress.py --config configs/stress_pinn.yaml` — обучение с нуля.